<a href="https://colab.research.google.com/github/nukui96/mackenzie-pa4-bss/blob/main/notebooks/01_cd_projeto_aplicado_IV_doc_etapa1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir este notebook no Colab"/></a>

<img src="https://raw.githubusercontent.com/scalabrinig/cdProjetoAplicadoIV/d093146488f56dfcf0ef286bcee8efe0e71b9c76/figuras/mackenzie_logo.jpg" width="25%" align="right"/>

# **PROJETO APLICADO IV - Tecnologia em Banco de Dados - 2026/02**

# **Documentação do Projeto**

---

> **Etapa 1:** Definição do Projeto e Equipe (Marco: 31/08/2026)

# **Título do Projeto**

## Previsão Diária de Demanda de Bicicletas Compartilhadas (London Bike Sharing)

---

In [ ]:
#@title **Identificação do Grupo**

#@markdown Integrantes do Grupo, nome completo em ordem alfabética (*informe: <nome>, <matrícula>*)
Aluno1 = 'Kayo Oliveira Nukui, 10356420' #@param {type:"string"}

# **Resumo da etapa 1**

Este projeto propõe investigar a previsão do total diário de novos aluguéis de bicicletas em Londres para o dia seguinte, usando um conjunto histórico público. Uma previsão com erro aceitável poderia apoiar decisões de planejamento agregado, mas essa utilidade ainda precisa ser avaliada. Nesta etapa, delimitamos o problema, os objetivos, a fonte dos dados e suas limitações iniciais. A escolha e a comparação dos métodos preditivos ocorrerão nas próximas etapas.


# **Introdução**

### Contexto e motivação
O compartilhamento de bicicletas é uma alternativa de deslocamento urbano. A demanda registrada pode variar com o calendário, as estações do ano e o clima. O conjunto London Bike Sharing permite estudar essas variações em dados horários históricos, sem representar a operação atual da cidade.

### Pergunta e objetivos
**Pergunta:** métodos de previsão simples conseguirão reduzir o erro em relação à repetição do mesmo dia da semana anterior, em uma avaliação temporal sem dados futuros?

**Objetivo geral:** prever o total diário de novos aluguéis em Londres para o dia seguinte (t+1).

**Objetivos específicos:** conferir a integridade da base, definir uma agregação diária justificada, construir uma referência sazonal e comparar técnicas em períodos posteriores aos usados no ajuste.

### Justificativa e ODS 11
Uma previsão diária confiável poderia apoiar o planejamento agregado de um serviço de bicicletas compartilhadas. A proposta tem relação temática com o ODS 11, especialmente a meta 11.2 sobre transporte sustentável (UNITED NATIONS, s.d.). Os dados históricos não demonstram impacto ambiental nem aplicação real do projeto.

### Base de dados
O [London Bike Sharing Dataset](https://www.kaggle.com/datasets/hmavrodiev/london-bike-sharing-dataset) reúne registros horários de novos aluguéis e variáveis meteorológicas e de calendário. O arquivo utilizado contém 17.414 registros, de 04/01/2015 a 03/01/2017, e 10 colunas. A contagem de aluguéis é `cnt`; `timestamp` indica a hora; `t1`, `t2`, `hum`, `wind_speed` e `weather_code` descrevem condições meteorológicas; `is_holiday`, `is_weekend` e `season` são indicadores de calendário. A página da fonte exige atribuição à Transport for London: “Powered by TfL Open Data”.


## Descrição inicial da base
Carregamos o arquivo original da TfL (`london_merged.csv`) para verificar dimensões, tipos de colunas, primeiras linhas e integridade (presença de valores nulos).

In [ ]:
import pandas as pd
import numpy as np
import os

# Localização do arquivo de dados brutos
caminho_dados = 'data/raw/london_merged.csv'
if not os.path.exists(caminho_dados):
    caminho_dados = '../data/raw/london_merged.csv'

# Leitura da base original
df = pd.read_csv(caminho_dados)

print(f"Dimensões da base: {df.shape[0]} linhas (registros horários) e {df.shape[1]} colunas.")
print("\nPrimeiras 5 linhas da base:")
df.head()

Dimensões da base: 17414 linhas (registros horários) e 10 colunas.

Primeiras 5 linhas da base:


,timestamp,cnt,t1,t2,hum,wind_speed,weather_code,is_holiday,is_weekend,season
0,2015-01-04 00:00:00,182,3.0,2.0,93.0,6.0,3.0,0.0,1.0,3.0
1,2015-01-04 01:00:00,138,3.0,2.5,93.0,5.0,1.0,0.0,1.0,3.0
2,2015-01-04 02:00:00,134,2.5,2.5,96.5,0.0,1.0,0.0,1.0,3.0
3,2015-01-04 03:00:00,72,2.0,2.0,100.0,0.0,1.0,0.0,1.0,3.0
4,2015-01-04 04:00:00,47,2.0,0.0,93.0,6.5,1.0,0.0,1.0,3.0


In [ ]:
# Verificação das colunas, tipos e valores ausentes
print("Informações estruturais e tipos:")
df.info()

print("\nContagem de valores ausentes (nulos) por coluna:")
print(df.isnull().sum())

Informações estruturais e tipos:
<class 'pandas.DataFrame'>
RangeIndex: 17414 entries, 0 to 17413
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   timestamp     17414 non-null  str    
 1   cnt           17414 non-null  int64  
 2   t1            17414 non-null  float64
 3   t2            17414 non-null  float64
 4   hum           17414 non-null  float64
 5   wind_speed    17414 non-null  float64
 6   weather_code  17414 non-null  float64
 7   is_holiday    17414 non-null  float64
 8   is_weekend    17414 non-null  float64
 9   season        17414 non-null  float64
dtypes: float64(8), int64(1), str(1)
memory usage: 1.3 MB

Contagem de valores ausentes (nulos) por coluna:
timestamp       0
cnt             0
t1              0
t2              0
hum             0
wind_speed      0
weather_code    0
is_holiday      0
is_weekend      0
season          0
dtype: int64


In [ ]:
# Resumo estatístico das variáveis numéricas
print("Estatísticas descritivas básicas:")
df[['cnt', 't1', 't2', 'hum', 'wind_speed']].describe()

Estatísticas descritivas básicas:


,cnt,t1,t2,hum,wind_speed
count,17414.000000,17414.000000,17414.000000,17414.000000,17414.000000
mean,1143.101642,12.468091,11.520836,72.324954,15.913063
std,1085.108068,5.571818,6.615145,14.313186,7.894570
min,0.000000,-1.500000,-6.000000,20.500000,0.000000
25%,257.000000,8.000000,6.000000,63.000000,10.000000
50%,844.000000,12.500000,12.500000,74.500000,15.000000
75%,1671.750000,16.000000,16.000000,83.000000,20.500000
max,7860.000000,34.000000,34.000000,100.000000,56.500000


### Limitações e observações iniciais

Os registros são horários, enquanto o objetivo é uma previsão diária. A agregação exigirá atenção à cobertura de cada data. Entre 04/01/2015 e 03/01/2017 há 731 dias calendário; uma grade simples de 24 posições por dia teria 17.544 posições, contra 17.414 registros presentes. A diferença de 130 posições não equivale automaticamente a falhas de coleta: o arquivo não declara fuso horário e uma transição de horário civil pode alterar a duração de um dia. As lacunas e os dias incompletos serão examinados antes de definir o alvo da modelagem.


# **Modelos**
*(Seção reservada para implementação na Etapa 3, conforme calendário acadêmico).*

# **Resultados**
*(Seção reservada para as Etapas 3 e 4, conforme calendário acadêmico).*

# **Dicussão e Conclusão**
*(Seção reservada para a entrega final - Etapa 4).*

# **Apresentação**
*(Seção reservada para o vídeo da entrega final - Etapa 4).*

# **Referências**

* HYNDMAN, R. J.; ATHANASOPOULOS, G. **Forecasting: Principles and Practice**. 3. ed. Melbourne: OTexts, 2021. Disponível em: <https://otexts.com/fpp3/>. Acesso em: 28 set. 2026.
* MAVRODIEV, H. **London Bike Sharing Dataset**. Kaggle, [s.d.]. Disponível em: <https://www.kaggle.com/datasets/hmavrodiev/london-bike-sharing-dataset>. Acesso em: 28 set. 2026.
* TRANSPORT FOR LONDON (TfL). **Open data users**. [s.d.]. Disponível em: <https://tfl.gov.uk/info-for/open-data-users/>. Acesso em: 28 set. 2026.
* UNITED NATIONS. **Goal 11: Sustainable Cities and Communities**. Department of Economic and Social Affairs, [s.d.]. Disponível em: <https://sdgs.un.org/goals/goal11>. Acesso em: 28 set. 2026.
